### Laboratorio 1 -- Series de Tiempo
**Análisis exploratorio — incisos d, e, f**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# El archivo .xlsx es realmente un CSV
df = pd.read_csv("./df/Base_Migracion_2009-2026jun.csv")
df['Viajero'] = pd.to_numeric(df['Viajero'], errors='coerce')

# Subconjunto consistente para todo el período 2009-2026
# entre 2022-2023 la categoría 'Viajero' excluyó viajeros no turísticos de alta frecuencia
# (comercio fronterizo, tránsito), por lo que solo Turista + Excursionista son comparables
df_te = df[df['Tipo de Viajero'].isin(['Turista', 'Excursionista'])].copy()

print(f"Total filas: {len(df)}, Turista+Excursionista: {len(df_te)}")
df.head(3)


### a,b,c) comportamiento temporal del número de viajeros, países con mayor cantidad de viajeros. regiones con mayor cantidad de viajeros

In [ ]:
#EXPLORACION INICIAL

print("\n INFORMACIÓN GENERAL")
print("-"*70)
print(f"Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"Años: {df['Año'].min()} - {df['Año'].max()}")
print(f"Tipos de viajero: {df['Tipo de Viajero'].unique().tolist()}")

print("\nTipos de df:")
print(df.dtypes)

print("\nValores nulos por columna:")
print(df.isnull().sum())

print("Estadistica descriptiva viajeros:")
print(df['Viajero'].describe().apply(lambda x: f'{x:,.0f}'))

print("Categorías únicas:")
print(f"Vias: {df['Vía'].unique()}")
print(f"Tipos de viajero: {df['Tipo de Viajero'].unique()}")
print(f"Años: {df['Año'].unique()}")


#a. comportamiento temporal del número de viajeros
df_filtracion = df[df['Tipo de Viajero'].isin(['Turista', 'Excursionista'])].copy()
df_filtracion['Fecha'] = pd.to_datetime(df_filtracion['Año'].astype(str) + '-' + df_filtracion['Mes cod'].astype(str)+'-01')

#tendencias: mensual y anual
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

mensual = df_filtracion.groupby('Fecha')['Viajero'].sum()
axes[0, 0].plot(mensual.index, mensual.values, color='blue', linewidth=1)
axes[0, 0].set_title('Evolución Mensual de Viajeros (Turista + Excursionista)', fontsize=12)
axes[0, 0].set_xlabel('Fecha')
axes[0, 0].set_ylabel('Número de Viajeros')
axes[0, 0].grid(True, alpha=0.3)

anual = df_filtracion.groupby('Año')['Viajero'].sum()
anual.index = anual.index.astype(int)
axes[0, 1].bar(anual.index, anual.values, color='skyblue')
axes[0, 1].set_title('Total Anual de Viajeros (Turista + Excursionista)', fontsize=12)
axes[0, 1].set_xlabel('Año')
axes[0, 1].set_ylabel('Número de Viajeros')
axes[0, 1].set_xticks(anual.index)
axes[0, 1].set_xticklabels(anual.index, rotation=45)
axes[0, 1].ticklabel_format(style='plain', axis='y', useOffset=False)
axes[0, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: format(int(x))))

estacionalidad = df_filtracion.groupby('Mes')['Viajero'].mean()
meses_orden = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
estacionalidad = estacionalidad.reindex(meses_orden)
axes[1, 0].plot(estacionalidad.index, estacionalidad.values, color='green')
axes[1, 0].set_title('Estacionalidad Mensual Promedio', fontsize=12)
axes[1, 0].set_xlabel('Mes')
axes[1, 0].set_ylabel('Promedio de Viajeros')
axes[1, 0].grid(True, alpha=0.3)

media_movil = mensual.rolling(window=12).mean()
axes[1, 1].plot(mensual.index, mensual.values, color='gray', alpha=0.5, label='df mensuales')
axes[1, 1].plot(media_movil.index, media_movil.values, color='red', linewidth=2, label='Media móvil 12 meses')
axes[1, 1].set_title('Tendencia con Media Móvil', fontsize=12)
axes[1, 1].set_xlabel('Fecha')
axes[1, 1].set_ylabel('Número de Viajeros')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#b. países con mayor cantidad de viajeros
top_paises = df_filtracion.groupby('País')['Viajero'].sum().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(15, 8))
bars = ax.barh(top_paises.index, top_paises.values, color='coral', edgecolor='darkred')
ax.set_title('Top 10 Países con Mayor Cantidad de Viajeros', fontsize=14)
ax.set_xlabel('Total de Viajeros', fontsize=12)
ax.invert_yaxis()

for i, (bar, value) in enumerate(zip(bars, top_paises.values)):
    ax.text(value, bar.get_y() + bar.get_height()/2, 
            f' {int(value):,}', va='center', ha='left', fontsize=9)

plt.tight_layout()
plt.show()

#c. regiones con mayor cantidad de viajeros
top_regiones = df_filtracion.groupby('Región dos')['Viajero'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))

# Crear barras horizontales
bars = ax.barh(top_regiones.index, top_regiones.values, color=plt.cm.Set2(np.linspace(0, 1, len(top_regiones))))

# Agregar valores y porcentajes al final de cada barra
total = top_regiones.sum()
for i, (bar, value) in enumerate(zip(bars, top_regiones.values)):
    pct = (value / total) * 100
    ax.text(value + (total * 0.01), bar.get_y() + bar.get_height()/2,
            f'{int(value):,} ({pct:.1f}%)',
            va='center', ha='left', fontsize=10)

ax.set_title('Distribución Viajeros por Región (Continentes)', fontsize=14, weight='bold')
ax.set_xlabel('Total de Viajeros', fontsize=12)
ax.set_ylabel('Región', fontsize=12)
ax.ticklabel_format(style='plain', axis='x')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: format(int(x), ',')))

plt.tight_layout()
plt.show()

### d) Vías de ingreso y fronteras más utilizadas

In [ ]:
# Vías de ingreso
vias = df['Vía'].value_counts()
print("=== VÍAS DE INGRESO ===")
print(vias)
print(f"\nPorcentajes:\n{(vias / vias.sum() * 100).round(2)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

vias.plot(kind='bar', ax=axes[0], color=['skyblue', 'salmon', 'lightgreen'])
axes[0].set_title('Frecuencia por Vía de Ingreso')
axes[0].set_ylabel('Cantidad de registros')
axes[0].tick_params(axis='x', rotation=0)

vias.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90,
          colors=['skyblue', 'salmon', 'lightgreen'], wedgeprops={'edgecolor': 'black'})
axes[1].set_title('Distribución Porcentual por Vía')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Fronteras más utilizadas
fronteras = df['Frontera'].value_counts()
print("=== TOP 10 FRONTERAS MÁS UTILIZADAS ===\n")
print(fronteras.head(10))
print(f"\nPorcentaje acumulado top 5: {(fronteras.head(5).sum() / fronteras.sum() * 100).round(2)}%")

plt.figure(figsize=(12, 6))
top15 = fronteras.head(15)
colors = plt.cm.Blues(np.linspace(0.4, 0.9, 15))
top15.plot(kind='barh', color=colors[::-1])
plt.title('Top 15 Fronteras más Utilizadas')
plt.xlabel('Cantidad de registros')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


**Interpretación:**

- La vía **terrestre** domina ampliamente (78.2% de los registros), seguida de aérea (15.5%) y marítima (6.3%).
- '01 La Aurora' (aeropuerto) es la frontera individual con más registros, pero las fronteras terrestres (Valle Nuevo, Melchor de Mencos, Pedro de Alvarado, El Florido) le siguen de cerca.
- Las 5 fronteras principales concentran ~46% del total de registros.


### e) Análisis de valores faltantes, duplicados y valores atípicos

In [ ]:
# Valores faltantes
print("=== VALORES FALTANTES ===")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(4)
missing_info = pd.DataFrame({'Faltantes': missing, 'Porcentaje': missing_pct})
print(missing_info[missing_info['Faltantes'] > 0] if missing.sum() > 0 else "No hay valores faltantes en ninguna columna.")

# Duplicados
dups = df.duplicated().sum()
print(f"\n=== DUPLICADOS ===")
print(f"Filas exactamente duplicadas: {dups}")

# Valores atípicos en Viajero (IQR)
print(f"\n=== VALORES ATÍPICOS (Viajero) ===")
Q1 = df['Viajero'].quantile(0.25)
Q3 = df['Viajero'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df['Viajero'] < lower) | (df['Viajero'] > upper)]
print(f"Q1 = {Q1:.2f}, Q3 = {Q3:.2f}, IQR = {IQR:.2f}")
print(f"Límite inferior = {lower:.2f}, Límite superior = {upper:.2f}")
print(f"Registros atípicos: {len(outliers)} ({len(outliers)/len(df)*100:.2f}% del total)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
axes[0].boxplot(df['Viajero'].clip(upper=df['Viajero'].quantile(0.99)), vert=False)
axes[0].set_title('Boxplot de Viajeros (recortado al percentil 99)')
axes[0].set_xlabel('Viajeros')

# Histograma
df['Viajero'].clip(upper=df['Viajero'].quantile(0.99)).hist(bins=50, ax=axes[1],
    color='coral', edgecolor='black', alpha=0.7)
axes[1].set_title('Distribución de Viajeros (percentil 99)')
axes[1].set_xlabel('Viajeros')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

**Interpretación:**

- El conjunto de df **no presenta valores faltantes ni filas duplicadas**, indicando que la base está completa y limpia.
- Sin embargo, ~16.4% de los registros se identifican como **valores atípicos** en 'Viajero' según el criterio IQR. Esto es esperable en df de migración: ciertas rutas/fronteras concentran volúmenes muy superiores al promedio.
- La distribución está **fuertemente sesgada a la derecha**: la mayoría de observaciones tienen pocos viajeros (mediana = 7), pero existen picos muy altos que elevan la media a ~325.


### f) Estadísticas descriptivas y visualizaciones con interpretación

In [ ]:
# Estadisticas descriptivas
print("=== ESTADÍSTICAS DESCRIPTIVAS (variable Viajero) ===")
desc = df['Viajero'].describe()
print(desc)
print(f"\nVarianza: {df['Viajero'].var():.2f}")
print(f"Sesgo (skewness): {df['Viajero'].skew():.2f}")
print(f"Curtosis: {df['Viajero'].kurtosis():.2f}")

# Estadísticas por vía de ingreso
print("\n=== ESTADÍSTICAS POR VÍA DE INGRESO ===")
print(df.groupby('Vía')['Viajero'].describe().round(2))

# Estadísticas por tipo de viajero
print("\n=== ESTADÍSTICAS POR TIPO DE VIAJERO ===")
print(df.groupby('Tipo de Viajero')['Viajero'].describe().round(2))

In [ ]:
# Visualizaciones (usando Turista+Excursionista para consistencia longitudinal)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Serie de tiempo mensual (Turista+Excursionista)
serie = df_te.groupby(['Año', 'Mes cod'])['Viajero'].sum().reset_index()
serie['fecha'] = pd.to_datetime(serie['Año'].astype(str) + '-' + serie['Mes cod'].astype(str))
serie = serie.sort_values('fecha')
axes[0, 0].plot(serie['fecha'], serie['Viajero'], color='steelblue', linewidth=1)
axes[0, 0].set_title('Evolución Mensual - Turista+Excursionista')
axes[0, 0].set_xlabel('Fecha')
axes[0, 0].set_ylabel('Viajeros')
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Top 10 países (Turista+Excursionista)
top_paises = df_te.groupby('País')['Viajero'].sum().sort_values(ascending=False).head(10)
colors = plt.cm.viridis(np.linspace(0.1, 0.9, 10))
top_paises.plot(kind='barh', ax=axes[0, 1], color=colors[::-1])
axes[0, 1].set_title('Top 10 Países - Turista+Excursionista')
axes[0, 1].set_xlabel('Total Viajeros')
axes[0, 1].invert_yaxis()

# 3. Distribución por década (Turista+Excursionista)
df_te['década'] = (df_te['Año'] // 10) * 10
df_te.boxplot(column='Viajero', by='década', ax=axes[1, 0],
              showfliers=False, patch_artist=True,
              boxprops=dict(facecolor='lightblue'))
axes[1, 0].set_title('Distribución por Década (sin outliers) - Turista+Excursionista')
axes[1, 0].set_xlabel('Década')
axes[1, 0].set_ylabel('Viajeros')

# 4. Promedio anual (Turista+Excursionista)
anual = df_te.groupby('Año')['Viajero'].mean()
axes[1, 1].scatter(anual.index, anual.values, color='darkorange', alpha=0.7, s=40)
z = np.polyfit(anual.index, anual.values, 1)
p = np.poly1d(z)
axes[1, 1].plot(anual.index, p(anual.index), 'r--', alpha=0.8)
axes[1, 1].set_title('Promedio Anual con Tendencia - Turista+Excursionista')
axes[1, 1].set_xlabel('Año')
axes[1, 1].set_ylabel('Promedio de Viajeros')

plt.suptitle('Análisis Exploratorio - Visualizaciones (Turista+Excursionista)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


**Interpretación general:**

1. La **serie de tiempo** (Turista+Excursionista) muestra una tendencia creciente sostenida entre 2009-2026, con una caída abrupta en 2020 (COVID-19) y recuperación posterior.
2. La distribución es altamente asimétrica (sesgo positivo): la mayoría de registros tienen valores bajos, con concentraciones masivas en ciertas rutas.
3. La **vía terrestre** es predominante (~78%), reflejando la geografía regional con fronteras porosas y alto tránsito vehicular.
4. '01 La Aurora' (aeropuerto) lidera en registros individuales, pero las fronteras terrestres en conjunto concentran el mayor volumen.
5. **Nota sobre la categoría 'Viajero':** Entre 2022-2023 se excluyó a viajeros no turísticos de alta frecuencia (comercio fronterizo, tránsito) de esta categoría. Por eso los totales que incluyen 'Viajero' muestran una caída artificial en 2023. Los análisis longitudinales usan **Turista + Excursionista**, que sí son consistentes en todo el período.


## Division entrenamiento y prueba

In [ ]:
#ENTRENAMIENTO Y PRUEBA
df_filtracion = df_filtracion.sort_values('Fecha').reset_index(drop=True)

train_size = int(len(df_filtracion)*0.7)
train_df = df_filtracion.iloc[:train_size] #70%
test_df = df_filtracion.iloc[train_size:] #30%

print("=== DIVISION DE DATOS ===")
print(f"Datos entrenamiento: {len(train_df)} ({len(train_df)/len(df_filtracion)*100:.1f}%)")
print(f"Datos prueba: {len(test_df)} ({len(test_df)/len(df_filtracion)*100:.1f}%)")


=== DIVISION DE DATOS ===
Datos entrenamiento: 96349 (70.0%)
Datos prueba: 41293 (30.0%)


## SERIES DE TIEMPO

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMAResults
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pmdarima import auto_arima